# Input

## Input: reusable reader

Java's input is slow by default, so every pattern below runs on one reusable reader that pulls bytes in big chunks and parses them itself. Once you have it, each input format is just a few lines. In C++ `cin` already skips whitespace for you, and this reader does the same.

```java
import java.io.*;
import java.util.*;

public class Main {
    static class FastScanner {
        private final InputStream in = System.in;
        private final byte[] buf = new byte[1 << 16];
        private int ptr = 0, len = 0;

        private int read() throws IOException {
            if (ptr == len) {
                len = in.read(buf, 0, buf.length);
                ptr = 0;
                if (len <= 0) { len = 0; return -1; }   // EOF
            }
            return buf[ptr++];
        }
        int nextInt() throws IOException { return (int) nextLong(); }
        long nextLong() throws IOException {
            int b = read();
            while (b != -1 && b != '-' && (b < '0' || b > '9')) b = read();
            boolean neg = false;
            if (b == '-') { neg = true; b = read(); }
            long r = 0;
            while (b >= '0' && b <= '9') { r = r * 10 + (b - '0'); b = read(); }
            return neg ? -r : r;
        }
        double nextDouble() throws IOException { return Double.parseDouble(next()); }
        String next() throws IOException {            // one whitespace-separated token
            int b = read();
            while (b != -1 && b <= ' ') b = read();
            StringBuilder sb = new StringBuilder();
            while (b > ' ') { sb.append((char) b); b = read(); }
            return sb.toString();
        }
        char nextChar() throws IOException {
            int b = read();
            while (b != -1 && b <= ' ') b = read();
            return (char) b;
        }
        String nextLine() throws IOException {         // rest of the current line
            StringBuilder sb = new StringBuilder();
            int b = read();
            while (b != -1 && b != '\n') { if (b != '\r') sb.append((char) b); b = read(); }
            return sb.toString();
        }
        int[] nextIntArray(int n) throws IOException {
            int[] a = new int[n];
            for (int i = 0; i < n; i++) a[i] = nextInt();
            return a;
        }
        long[] nextLongArray(int n) throws IOException {
            long[] a = new long[n];
            for (int i = 0; i < n; i++) a[i] = nextLong();
            return a;
        }
    }

    public static void main(String[] args) throws IOException {
        FastScanner sc = new FastScanner();
        PrintWriter out = new PrintWriter(new BufferedWriter(new OutputStreamWriter(System.out)));
        // ... solve ...
        out.flush();
    }
}
```

The `throws IOException` is needed because reading can fail. Putting it on `main` means you never write try/catch.

## Input: common formats

| Input format | C++ | Java (with `sc` above) |
|---|---|---|
| One int / long / double | `int x; long long y; double d; cin >> x >> y >> d;` | `int x = sc.nextInt(); long y = sc.nextLong(); double d = sc.nextDouble();` |
| One word | `string s; cin >> s;` | `String s = sc.next();` |
| One char | `char c; cin >> c;` | `char c = sc.nextChar();` |
| `n` then `n` numbers | `int n; cin >> n; vector<int> a(n); for (auto& x : a) cin >> x;` | `int n = sc.nextInt(); int[] a = sc.nextIntArray(n);` |
| `n x m` int matrix | `vector<vector<int>> g(n, vector<int>(m)); for (int i = 0; i < n; i++) for (int j = 0; j < m; j++) cin >> g[i][j];` | `int[][] g = new int[n][m]; for (int i = 0; i < n; i++) for (int j = 0; j < m; j++) g[i][j] = sc.nextInt();` |
| Grid of chars (rows like `#..#`) | `vector<string> g(n); for (auto& r : g) cin >> r;` | `char[][] g = new char[n][]; for (int i = 0; i < n; i++) g[i] = sc.next().toCharArray();` |
| Line with spaces | `string s; getline(cin, s);` | `String s = sc.nextLine();` |
| Edges, 1-indexed | `int a, b; cin >> a >> b; a--; b--;` | `int a = sc.nextInt() - 1, b = sc.nextInt() - 1;` |
| Multiple test cases | `int t; cin >> t; while (t--) { int n; cin >> n; }` | `int t = sc.nextInt(); while (t-- > 0) { int n = sc.nextInt(); }` |
| Queries `q` then `q` lines | `int q; cin >> q; while (q--) { int type, x; cin >> type >> x; }` | `int q = sc.nextInt(); while (q-- > 0) { int type = sc.nextInt(), x = sc.nextInt(); }` |
| Read until EOF | `int x; while (cin >> x) { }` | `String s; while ((s = br.readLine()) != null) { }` (`BufferedReader br`, see the `BufferedReader` fallback below) |
| Pair of numbers | `pair<int,int> p; cin >> p.first >> p.second;` | `int[] p = {sc.nextInt(), sc.nextInt()};` |

**Mixing `nextLine` with other reads:** after `sc.nextInt()`, the newline is still unread, so a following `nextLine()` returns an empty string. Call `nextLine()` once to throw that away. C++ `getline` has the same trap after `cin >>`.

**Reading until EOF:** `nextInt` returns 0 at EOF, which can't be told apart from a real 0. If a problem says "until end of input", use the `BufferedReader` approach in the `BufferedReader` fallback section below.

## Input: graphs

```java
// Adjacency list, directed, n vertices, m edges
List<Integer>[] g = new ArrayList[n];
for (int i = 0; i < n; i++) g[i] = new ArrayList<>();
for (int i = 0; i < m; i++) {
    int a = sc.nextInt() - 1, b = sc.nextInt() - 1;
    g[a].add(b);
    // g[b].add(a);   // add this line for undirected
}

// Weighted: store {to, weight}
List<int[]>[] g = new ArrayList[n];
for (int i = 0; i < n; i++) g[i] = new ArrayList<>();
for (int i = 0; i < m; i++) {
    int a = sc.nextInt() - 1, b = sc.nextInt() - 1, w = sc.nextInt();
    g[a].add(new int[]{b, w});
}

// Tree with n-1 edges: same loop, run n-1 times
```

## Input: `BufferedReader` fallback

Slower than the byte reader but simpler, and the right tool for EOF and line-oriented input.

```java
BufferedReader br = new BufferedReader(new InputStreamReader(System.in));

String line = br.readLine();                     // null at EOF
StringTokenizer st = new StringTokenizer(line);
int n = Integer.parseInt(st.nextToken());
int m = Integer.parseInt(st.nextToken());

// Read until EOF
String s;
while ((s = br.readLine()) != null) {
    // process one line
}

// Whole line as an int array
int[] a = Arrays.stream(br.readLine().trim().split(" ")).mapToInt(Integer::parseInt).toArray();
```

The `split(" ")` approach breaks if there are double spaces or trailing spaces. `StringTokenizer` handles those safely.

### Parsing structured lines with regex

For lines like `turn on 0,0 through 999,999`, a `Pattern` with capture groups beats `split` and `substring`. `import java.util.regex.*;`

```java
private static final Pattern INSTRUCTION =
        Pattern.compile("(turn on|turn off|toggle) (\\d+),(\\d+) through (\\d+),(\\d+)");

public void apply(String instruction) {
    Matcher m = INSTRUCTION.matcher(instruction);
    if (!m.matches()) throw new IllegalArgumentException(instruction);

    String action = m.group(1);
    int i1 = Integer.parseInt(m.group(2));
    int j1 = Integer.parseInt(m.group(3));
    int i2 = Integer.parseInt(m.group(4));
    int j2 = Integer.parseInt(m.group(5));

    switch (action) {
        case "turn on"  -> { /* ... */ }
        case "turn off" -> { /* ... */ }
        case "toggle"   -> { /* ... */ }
    }
}
```

Group 0 is the whole match and groups 1 onward are the parentheses in order. `matches()` requires the whole string to fit the pattern, while `find()` looks for a match anywhere inside it. Compile the `Pattern` once as a `static final`, not inside the per-line method. The arrow `case` syntax needs Java 14 or later, so use `case "toggle":` with `break;` on older judges.

## Which reader to pick

- **Default:** the `FastScanner` above, which handles up to about 10^6 numbers comfortably.
- **Line-oriented or EOF-terminated input:** `BufferedReader`.
- **Interactive problems:** use `BufferedReader` and flush `System.out` after every query, since the byte-buffer reader may block waiting to fill its chunk.
- **Never use `Scanner`:** it can be 5-10x slower and causes time-limit failures at around 10^5 numbers.

## C++ to Java I/O quick map

Every row shows a concrete example in both columns, so you can copy either side as is.

| C++ | Java |
|---|---|
| `int x; cin >> x;` | `int x = sc.nextInt();` (custom reader, `FastScanner` above) |
| `cout << x << '\n';` | `out.println(x);` (`PrintWriter`) |
| `string out; out += to_string(x) + "\n"; cout << out;` | `StringBuilder sb = new StringBuilder(); sb.append(x).append('\n'); out.print(sb);` |
| `printf("%.10f\n", d);` | `out.printf("%.10f%n", d);` or `System.out.printf("%.10f%n", d);` |